In [1]:
%%capture
!pip uninstall -y transformers tokenizers
!pip install transformers==4.44.2 tokenizers==0.19.1 datasets==2.19.0

In [5]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
import os
import math
import torch
import random
import numpy as np
from datasets import load_dataset
from transformers import (
    RoFormerConfig, RoFormerForMaskedLM, RobertaTokenizerFast,
    Trainer, TrainingArguments
)

In [ ]:
BASE_DIR = "/content/drive/MyDrive/AncientRusProject_RoFormer_V2"
DATA_FILE = f"{BASE_DIR}/ancient_rus_ready_for_bert.txt"
TOKENIZER_DIR = f"{BASE_DIR}/ancient_rus_tokenizer_BPE"
MODEL_DIR = f"{BASE_DIR}/mini_roformer_ancient_rus"
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
print("BPE tokenizator loading...")
tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_DIR, max_len=512)

⏳ Загрузка BPE токенизатора...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
special_tokens_dict = {
    'additional_special_tokens': [
        "[CTX_CHURCH]", "[CTX_DAILY]", "[CTX_LEGAL]",
        "[CTX_LIT]", "[CTX_EPIC]", "[CTX_SCIENCE]", "[GAP]"
    ]
}

In [ ]:
tokenizer.add_special_tokens(special_tokens_dict)

7

In [ ]:
dataset = load_dataset("text", data_files={"train": DATA_FILE})
split_dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=False)

In [ ]:
tokenized_datasets = split_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

In [ ]:
def group_texts(examples):
    block_size = 256
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated_examples[list(examples.keys())[0]]) // block_size) * block_size
    return {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }

In [ ]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)

In [ ]:
class PhysicalDegradationCollator:
    """
    Simulates real damage to historical documents:
    1. Broken edges (Edge Masking)
    2. Worn holes (Span Masking)
    3. Erased parts of the words (Random Subword Masking)
    """
    def __init__(self, tokenizer, mlm_prob=0.15, max_span=3, edge_prob=0.1):
        self.tokenizer = tokenizer
        self.mlm_prob = mlm_prob
        self.max_span = max_span
        self.edge_prob = edge_prob

    def __call__(self, features):
        input_ids = torch.tensor([f["input_ids"] for f in features], dtype=torch.long)
        attention_mask = torch.tensor([f["attention_mask"] for f in features], dtype=torch.long)
        labels = input_ids.clone()

        batch_size, seq_len = input_ids.shape
        probability_matrix = torch.full(labels.shape, self.mlm_prob)

        # Special tokens protection
        special_tokens_mask = [
            self.tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) for val in labels.tolist()
        ]
        special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)

        # Basic random mask
        masked_indices = torch.bernoulli(probability_matrix).bool()
        final_mask = masked_indices.clone()



        for i in range(batch_size):
            # 1. Edge Masking
            if random.random() < self.edge_prob:
                edge_len = random.randint(2, 5)
                is_start = random.choice([True, False])

                # Looking for boundaries, ignoring <s> and </s> и and the context tags
                valid_indices = (~special_tokens_mask[i]).nonzero(as_tuple=True)[0]
                if len(valid_indices) > edge_len:
                    if is_start:
                        start_idx = valid_indices[0]
                        final_mask[i, start_idx : start_idx + edge_len] = True
                    else:
                        end_idx = valid_indices[-1]
                        final_mask[i, end_idx - edge_len + 1 : end_idx + 1] = True

            # 2. Span Masking
            for j in range(seq_len):
                if masked_indices[i, j]:
                    span_len = random.randint(1, self.max_span)
                    end_idx = min(j + span_len, seq_len)
                    if not special_tokens_mask[i, j:end_idx].any():
                        final_mask[i, j:end_idx] = True

        labels[~final_mask] = -100

        # Standard 80% [MASK], 10% random, 10% original
        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & final_mask
        input_ids[indices_replaced] = self.tokenizer.mask_token_id

        indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & final_mask & ~indices_replaced
        random_words = torch.randint(len(self.tokenizer), labels.shape, dtype=torch.long)
        input_ids[indices_random] = random_words[indices_random]

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
data_collator = PhysicalDegradationCollator(tokenizer=tokenizer, mlm_prob=0.12, max_span=3, edge_prob=0.15)

In [ ]:
config = RoFormerConfig(
    vocab_size=len(tokenizer),
    embedding_size=512,
    hidden_size=512,
    num_hidden_layers=6,
    num_attention_heads=8,
    intermediate_size=2048,
    max_position_embeddings=514,
    pad_token_id=tokenizer.pad_token_id,
    rotary_value=False
)

In [ ]:
model = RoFormerForMaskedLM(config)
model.resize_token_embeddings(len(tokenizer))
print(f"Mini-RoFormer parameters: {model.num_parameters():,}")

🧠 Параметры Mini-RoFormer: 44,862,928


In [ ]:
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return torch.topk(logits, k=5, dim=-1).indices

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    mask = labels != -100
    labels = labels[mask]
    preds = preds[mask]

    return {
        "top1_accuracy": np.mean(preds[:, 0] == labels),
        "top3_accuracy": np.mean(np.any(preds[:, :3] == labels[:, None], axis=1)),
        "top5_accuracy": np.mean(np.any(preds[:, :5] == labels[:, None], axis=1)),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    overwrite_output_dir=True,
    num_train_epochs=15,


    per_device_train_batch_size=32,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=32,

    evaluation_strategy="steps",
    eval_steps=400,
    save_steps=400,
    save_total_limit=2,
    logging_steps=100,
    prediction_loss_only=False,

    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=1000,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
    load_best_model_at_end=True
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Top1 Accuracy,Top3 Accuracy,Top5 Accuracy
400,7.320200,7.088848,0.188519,0.252550,0.281513
800,6.133500,5.925769,0.247061,0.319558,0.350575
1200,5.306700,5.143108,0.291822,0.375850,0.411623
1600,4.692100,4.539958,0.341675,0.438206,0.477252
2000,4.232200,4.104209,0.385696,0.484114,0.524598
2400,3.906100,3.787649,0.421273,0.522242,0.562056
2800,3.654200,3.545280,0.450474,0.551822,0.591808
3200,3.436200,3.347799,0.473913,0.575595,0.614161
3600,3.230400,3.154341,0.498776,0.600539,0.637544
4000,3.117500,3.032521,0.514085,0.615329,0.652676


There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


TrainOutput(global_step=8760, training_loss=3.595059841626311, metrics={'train_runtime': 10589.7208, 'train_samples_per_second': 105.99, 'train_steps_per_second': 0.827, 'total_flos': 3.310916929825997e+16, 'train_loss': 3.595059841626311, 'epoch': 14.980761008978195})

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

('/content/drive/MyDrive/AncientRusProject_RoFormer_V2/mini_roformer_ancient_rus/tokenizer_config.json',
 '/content/drive/MyDrive/AncientRusProject_RoFormer_V2/mini_roformer_ancient_rus/special_tokens_map.json',
 '/content/drive/MyDrive/AncientRusProject_RoFormer_V2/mini_roformer_ancient_rus/vocab.json',
 '/content/drive/MyDrive/AncientRusProject_RoFormer_V2/mini_roformer_ancient_rus/merges.txt',
 '/content/drive/MyDrive/AncientRusProject_RoFormer_V2/mini_roformer_ancient_rus/added_tokens.json',
 '/content/drive/MyDrive/AncientRusProject_RoFormer_V2/mini_roformer_ancient_rus/tokenizer.json')

In [ ]:
print("\nFINAL RESULTS:")
eval_results = trainer.evaluate()
print(f"Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")
print(f"Top-1 Accuracy: {eval_results.get('eval_top1_accuracy', 0):.2%}")
print(f"Top-3 Accuracy: {eval_results.get('eval_top3_accuracy', 0):.2%}")
print(f"Top-5 Accuracy: {eval_results.get('eval_top5_accuracy', 0):.2%}")


📊 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:


Loss: 2.3729
Perplexity: 10.73
Top-5 Точность: 73.03%


In [ ]:
from transformers import pipeline

In [ ]:
roformer_pipe = pipeline(
    "fill-mask",
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    device=0 # If GPU is available, else -1
)

In [ ]:

test_cases = [
    # 1. Classic: checking cases and logic (Chronicles)
    {
        "desc": "📚 Летописи (на какую землю?)",
        "text": "[CTX_LIT] И пошелъ князь игорь на <mask> землю со своею дружиною.",
        "expected": "рускую / свою"
    },

    # 2. Sudebnic: testing knowledge of specific laws
    {
        "desc": "⚖️ Русская Правда (кого убили?)",
        "text": "[CTX_LEGAL] Аже кто оубиеть <mask> , то платити виру 40 гривенъ.",
        "expected": "мужь"
    },

    # 3. Daily: checking understanding of debts
    {
        "desc": "🏡 Грамоты (про что пишут?)",
        "text": "[CTX_DAILY] поклоне ѿ ꙩндреꙗ · к ѥва · и к микифору про <mask> ѡкупи ꙩсподине",
        "expected": "серебро"
    },

    # 4. Tear-off edge testing (Edge Masking)
    # ​The sentence has no beginning, but RoPE must understand that a bow is being sent to Vasily
    {
        "desc": "🧨 ТЕСТ ROPE: Оторванное начало",
        "text": "[CTX_DAILY] <mask> <mask> ко василью . а серебро ми отдай.",
        "expected": "поклонъ ѿ"
    },

    # 5. Tear-off edge testing (Edge Masking)
    {
        "desc": "🧨 ТЕСТ ROPE: Оторванный конец",
        "text": "[CTX_EPIC] Выезжал добрый <mask> из <mask> на <mask> <mask>",
        "expected": "молодец из города на добром коне"
    },

    # 6. [GAP] testing
    {
        "desc": "🧩 ТЕСТ [GAP]: Работа с нечитаемым текстом",
        "text": "[CTX_DAILY] [GAP] бь ѿ но [GAP] тию и св <mask> коуно",
        "expected": "Модель должна предложить варианты, игнорируя дыры [GAP]"
    },

    # 7. Church: plural check
    {
        "desc": "⛪️ Церковный (кому сказал?)",
        "text": "[CTX_CHURCH] И рече господь къ <mask> своимъ, глаголя...",
        "expected": "ученикомъ / людемъ"
    }
]

print("\n" + "=" * 60)
print(" Roformer testing (RoPE + Phisical Degradation collator)")
print("=" * 60)

for idx, case in enumerate(test_cases, 1):
    print(f"\n[{idx}/7] {case['desc']}")
    print(f"Text: {case['text']}")
    print(f"Expected (meaning): {case['expected']}")

    # If there are more than 5 masks
    mask_count = case['text'].count("<mask>")
    results = roformer_pipe(case['text'], top_k=3)

    # Output normalization
    if mask_count == 1:
        results = [results]

    for i, mask_res in enumerate(results):
        print(f"Mask {i+1}: ", end="")
        preds = []
        for res in mask_res:
            clean_word = res['token_str'].replace("Ġ", "").strip()
            score = res['score'] * 100
            preds.append(f"'{clean_word}' ({score:.1f}%)")
        print(" | ".join(preds))


🚀 СТАРТ ТЕСТИРОВАНИЯ ROFORMER (RoPE + DEGRADATION)

[1/7] 📚 Летописи (на какую землю?)
📝 Текст: [CTX_LIT] И пошелъ князь игорь на <mask> землю со своею дружиною.
🎯 Ожидалось (смысл): рускую / свою
  ➡️ Маска 1: 'Русскую' (66.9%) | 'свою' (6.3%) | 'рускую' (4.2%)

[2/7] ⚖️ Русская Правда (кого убили?)
📝 Текст: [CTX_LEGAL] Аже кто оубиеть <mask> , то платити виру 40 гривенъ.
🎯 Ожидалось (смысл): мужь
  ➡️ Маска 1: 'мужь' (16.3%) | 'княжа' (6.0%) | 'звѣрь' (4.2%)

[3/7] 🏡 Грамоты (про что пишут?)
📝 Текст: [CTX_DAILY] поклоне ѿ ꙩндреꙗ · к ѥва · и к микифору про <mask> ѡкупи ꙩсподине
🎯 Ожидалось (смысл): серебро
  ➡️ Маска 1: '·' (93.9%) | 'серебро' (5.8%) | 'рубль' (0.0%)

[4/7] 🧨 ТЕСТ ROPE: Оторванное начало
📝 Текст: [CTX_DAILY] <mask> <mask> ко василью . а серебро ми отдай.
🎯 Ожидалось (смысл): поклонъ ѿ
  ➡️ Маска 1: 'От' (45.4%) | 'При' (8.6%) | 'Покланяние' (7.4%)
  ➡️ Маска 2: 'ныне' (16.6%) | 'игната' (7.6%) | 'ми' (5.6%)

[5/7] 🧨 ТЕСТ ROPE: Оторванный конец
📝 Текст: [CTX_EPIC] Вые

In [ ]:
import os
print(os.listdir(MODEL_DIR))

['checkpoint-8400', 'checkpoint-8760', 'config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'training_args.bin']


In [ ]:
%%capture
!pip install huggingface_hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
repo_id = "AlexSychovUN/mini-roformer-ancient-rus-v2"

In [ ]:
model.push_to_hub(repo_id, commit_message="Initial commit: Model weights")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...y87vcq8/model.safetensors:   0%|          |  557kB /  179MB            

CommitInfo(commit_url='https://huggingface.co/AlexSychovUN/mini-roformer-ancient-rus-v2/commit/3586861fdc0b3f9df9cb66acf0c674da059bc88e', commit_message='Initial commit: Model weights', commit_description='', oid='3586861fdc0b3f9df9cb66acf0c674da059bc88e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/AlexSychovUN/mini-roformer-ancient-rus-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='AlexSychovUN/mini-roformer-ancient-rus-v2'), pr_revision=None, pr_num=None)

In [ ]:
tokenizer.push_to_hub(repo_id, commit_message="Initial commit: Custom BPE Tokenizer")

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/AlexSychovUN/mini-roformer-ancient-rus-v2/commit/aca3c2b76e004706860d8103ce0abbd57cba982a', commit_message='Initial commit: Custom BPE Tokenizer', commit_description='', oid='aca3c2b76e004706860d8103ce0abbd57cba982a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/AlexSychovUN/mini-roformer-ancient-rus-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='AlexSychovUN/mini-roformer-ancient-rus-v2'), pr_revision=None, pr_num=None)